In [3]:
import segmentation_models_pytorch as smp
from torchinfo import summary

In [ ]:
model = smp.Unet(
    encoder_name="efficientnet-b7",
    encoder_weights="imagenet",
    in_channels=3,
    classes=6,
)
summary(model, input_size=(1, 3, 256, 256))

Layer (type:depth-idx)                                  Output Shape              Param #
Unet                                                    [1, 6, 256, 256]          --
├─EfficientNetEncoder: 1-1                              [1, 3, 256, 256]          1,643,520
│    └─Conv2dStaticSamePadding: 2-1                     [1, 64, 128, 128]         1,728
│    │    └─ZeroPad2d: 3-1                              [1, 3, 257, 257]          --
│    └─BatchNorm2d: 2-2                                 [1, 64, 128, 128]         128
│    └─SiLU: 2-3                                        [1, 64, 128, 128]         --
│    └─ModuleList: 2-4                                  --                        49,823,608
│    │    └─MBConvBlock: 3-2                            [1, 32, 128, 128]         4,944
│    │    └─MBConvBlock: 3-3                            [1, 32, 128, 128]         1,992
│    │    └─MBConvBlock: 3-4                            [1, 32, 128, 128]         1,992
│    │    └─MBConvBlock: 3-5    

In [8]:
from transformers import UperNetForSemanticSegmentation

id2label = {
    0: "impervious_surface",
    1: "building",
    2: "low_vegetation",
    3: "tree",
    4: "car",
    5: "clutter",
}

label2id = {v: k for k, v in id2label.items()}

model = UperNetForSemanticSegmentation.from_pretrained(
    "openmmlab/upernet-swin-tiny",
    num_labels=6,
    ignore_mismatched_sizes=True,
)
summary(model, input_size=(1, 3, 256, 256))

## backbones:
# "openmmlab/upernet-swin-small"
# "openmmlab/upernet-swin-base"

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/307 [00:00<?, ?it/s]

[transformers] UperNetForSemanticSegmentation LOAD REPORT from: openmmlab/upernet-swin-tiny
Key                              | Status   |                                                                                                     
---------------------------------+----------+-----------------------------------------------------------------------------------------------------
auxiliary_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([6])                      
decode_head.classifier.weight    | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 512, 1, 1]) vs model:torch.Size([6, 512, 1, 1])
auxiliary_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([6, 256, 1, 1])
decode_head.classifier.bias      | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([6])                      

Notes:
- MISMATCH:	ckpt w

Layer (type:depth-idx)                                                           Output Shape              Param #
UperNetForSemanticSegmentation                                                   [1, 6, 256, 256]          --
├─SwinBackbone: 1-1                                                              [1, 96, 64, 64]           --
│    └─SwinModel: 2-1                                                            [1, 96, 64, 64]           --
│    │    └─SwinEmbeddings: 3-1                                                  [1, 4096, 96]             4,896
│    │    └─SwinEncoder: 3-2                                                     [1, 4096, 96]             27,512,922
│    │    └─LayerNorm: 3-3                                                       [1, 64, 768]              1,536
│    └─ModuleDict: 2-2                                                           --                        --
│    │    └─LayerNorm: 3-4                                                       [1, 4096, 96]       

In [ ]:
import torch

x = torch.randn(1, 3, 256, 256).to("cuda")
output = model(x)

tensor([1, 2, 3, 5], device='cuda:0')

In [ ]:
from transformers import Mask2FormerForUniversalSegmentation, AutoImageProcessor

MODEL_NAME = "facebook/mask2former-swin-tiny-ade-semantic"
image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = Mask2FormerForUniversalSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    ignore_mismatched_sizes=True,
)
summary(model, input_size=(1, 3, 256, 256))

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/554 [00:00<?, ?it/s]

[transformers] Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-ade-semantic
Key                                                                                                                                | Status     |                                                                                         
-----------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.pixel_level_module.encoder.swin.encoder.layers.{0, 1, 2, 3}.blocks.{0, 1, 2, 3, 4, 5}.attention.self.relative_position_index | UNEXPECTED |                                                                                         
model.pixel_level_module.encoder.swin.layernorm.weight                                                                             | MISSING    |                                     

Layer (type:depth-idx)                                                                                    Output Shape              Param #
Mask2FormerForUniversalSegmentation                                                                       [1, 100, 256]             --
├─Mask2FormerModel: 1-1                                                                                   [1, 100, 64, 64]          --
│    └─Mask2FormerPixelLevelModule: 2-1                                                                   [1, 256, 8, 8]            --
│    │    └─SwinBackbone: 3-1                                                                             [1, 96, 64, 64]           27,522,234
│    │    └─Mask2FormerPixelDecoder: 3-2                                                                  [1, 256, 64, 64]          5,421,504
│    └─Mask2FormerTransformerModule: 2-2                                                                  [100, 1, 256]             51,968
│    │    └─Mask2FormerSinePosi

In [49]:
x = torch.randn(1, 3, 256, 256).to("cuda")
outputs = model(x)

processor_output = image_processor.post_process_semantic_segmentation(
    outputs, target_sizes=[(256, 256)] * 1
)[0].unsqueeze(0)
mask_logits = torch.nn.functional.interpolate(
    outputs.masks_queries_logits,
    size=(256, 256),
    mode="bilinear",
    align_corners=False,
)

class_logits = outputs.class_queries_logits[..., :-1]

masks_classes = class_logits.softmax(dim=-1)

semantic_logits = torch.einsum(
    "bqc,bqhw->bchw",
    masks_classes,
    mask_logits.sigmoid(),
)

preds = semantic_logits.argmax(dim=1)
torch.equal(processor_output, preds)

False

In [46]:
processor_output.shape, preds.shape

(torch.Size([1, 256, 256]), torch.Size([1, 256, 256]))

In [47]:
processor_output

tensor([[[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]]], device='cuda:0')

In [ ]:
agreement = (preds == processor_output).float().mean()
print(agreement)

tensor(0.9966, device='cuda:0')


In [1]:
import segmentation_models_pytorch as smp
from torchinfo import summary

model = smp.UPerNet(
    encoder_name="tu-convnext_small",
    encoder_weights="imagenet",
    in_channels=3,
    classes=6,
)
summary(model, input_size=(1, 3, 256, 256))

model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

Layer (type:depth-idx)                                  Output Shape              Param #
UPerNet                                                 [1, 6, 256, 256]          --
├─TimmUniversalEncoder: 1-1                             [1, 3, 256, 256]          --
│    └─FeatureListNet: 2-1                              [1, 96, 64, 64]           --
│    │    └─Conv2d: 3-1                                 [1, 96, 64, 64]           4,704
│    │    └─LayerNorm2d: 3-2                            [1, 96, 64, 64]           192
│    │    └─ConvNeXtStage: 3-3                          [1, 96, 64, 64]           237,888
│    │    └─ConvNeXtStage: 3-4                          [1, 192, 32, 32]          992,256
│    │    └─ConvNeXtStage: 3-5                          [1, 384, 16, 16]          32,747,520
│    │    └─ConvNeXtStage: 3-6                          [1, 768, 8, 8]            15,470,592
├─UPerNetDecoder: 1-2                                   [1, 256, 64, 64]          --
│    └─ModuleList: 2-2        

In [35]:
model = smp.Segformer(
    encoder_name="efficientnet-b6", encoder_weights="imagenet", in_channels=3, classes=6
)
summary(model, input_size=(1, 3, 256, 256))

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/173M [00:00<?, ?B/s]

Layer (type:depth-idx)                                  Output Shape              Param #
Segformer                                               [1, 6, 256, 256]          --
├─EfficientNetEncoder: 1-1                              [1, 3, 256, 256]          1,331,712
│    └─Conv2dStaticSamePadding: 2-1                     [1, 56, 128, 128]         1,512
│    │    └─ZeroPad2d: 3-1                              [1, 3, 257, 257]          --
│    └─BatchNorm2d: 2-2                                 [1, 56, 128, 128]         112
│    └─SiLU: 2-3                                        [1, 56, 128, 128]         --
│    └─ModuleList: 2-4                                  --                        --
│    │    └─MBConvBlock: 3-2                            [1, 32, 128, 128]         4,110
│    │    └─MBConvBlock: 3-3                            [1, 32, 128, 128]         1,992
│    │    └─MBConvBlock: 3-4                            [1, 32, 128, 128]         1,992
│    │    └─MBConvBlock: 3-5            

In [33]:
model = smp.Segformer(
    encoder_name="mit_b4", encoder_weights="imagenet", in_channels=3, classes=6
)
summary(model, input_size=(1, 3, 256, 256))

config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

Layer (type:depth-idx)                        Output Shape              Param #
Segformer                                     [1, 6, 256, 256]          --
├─MixVisionTransformerEncoder: 1-1            [1, 3, 256, 256]          --
│    └─OverlapPatchEmbed: 2-1                 [1, 64, 64, 64]           --
│    │    └─Conv2d: 3-1                       [1, 64, 64, 64]           9,472
│    │    └─LayerNorm: 3-2                    [1, 64, 64, 64]           128
│    └─Sequential: 2-2                        [1, 64, 64, 64]           --
│    │    └─Block: 3-3                        [1, 64, 64, 64]           314,880
│    │    └─Block: 3-4                        [1, 64, 64, 64]           314,880
│    │    └─Block: 3-5                        [1, 64, 64, 64]           314,880
│    └─LayerNorm: 2-3                         [1, 64, 64, 64]           128
│    └─OverlapPatchEmbed: 2-4                 [1, 128, 32, 32]          --
│    │    └─Conv2d: 3-6                       [1, 128, 32, 32]          73,

In [1]:
import requests


def notify(msg):
    requests.post("https://ntfy.sh/semantic-segmentation", data=msg)


notify("Testing Topic")